# 01 - Quick Start and Core Workflow

> **One line of code, ten thousand rows of data. Zero-config smart generation, AI-driven precise tuning.**

## Scenario Navigation

| My Need | Recommended Notebook | Core API |
|----------|---------------|----------|
| Quickly generate test data | **01 - Quick Start** ← you are here | `fill()` |
| Customize data type per column | 02 - Column Mapping | `columns={}` |
| Choose data generation engine | 03 - Generators and Providers | `provider=` |
| Multi-table relations, FK integrity | 04 - Database and Multi-table | `connect()` + `fill_from_config()` |
| Derivation between columns | 05 - Expressions and Constraints | `derive_from` + `expression` |
| YAML config-driven, batch generation | 06 - Config and Transform | `fill_from_config()` |
| AI auto-generates config | 07 - AI Smart Config | `sqlseed-ai` plugin |
| AI assistant operates database | 08 - MCP Server | `mcp-server-sqlseed` |
| Custom plugin extensions | 09 - Plugins and Hooks | `pluggy` |
| Command-line operations | 10 - CLI Reference | `sqlseed` CLI |

## What You Will Learn

- One-line fill: the power of `sqlseed.fill()`
- Zero-config smart inference: how sqlseed auto-selects generators
- Preview data: `sqlseed.preview()` without writing to DB
- Context manager: `sqlseed.connect()` for fine-grained control
- Core parameters: count, provider, seed, batch_size, enrich

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| **→ 01** | **Quick Start and Core Workflow** | **Orchestrator** | **None** |
| 02 | 9-Level Strategy Chain | Core: ColumnMapper | 01 |
| 03 | Generators and Provider System | Generators | 01 |
| 04 | Database Layer and Multi-table | Database + Core | 01 |
| 05 | Expression Derivation and Constraint Solving | Core: DAG / Expression | 01 |
| 06 | Config-Driven and Transform | Config / Core | 01 |
| 07 | AI Smart Config | Plugins: AI | 01 |
| 08 | MCP Server Integration | Plugins: MCP | 07 |
| 09 | Plugin System and Hook Lifecycle | Plugins | 01 |
| 10 | CLI Reference Manual | CLI | 06 |
| 11 | Utilities Reference | Utils | 01 |
| 12 | Testing Integration Patterns | Testing | 01 |

---
## Setup

Use Python 3.10+ and run this notebook from `examples/notebooks` in a repository checkout. Select a notebook kernel from the environment containing these packages:

```bash
python -m pip install 'sqlseed[mimesis]==0.2.4' 'sqlseed-cli==0.2.4' jupyterlab
```

For source development, install Core and CLI together as described in the [repository README](../../README.md). This notebook creates its own temporary database and cache. Run cells from top to bottom; the validation helpers raise on partial generation or failed CLI commands.


In [ ]:
PRJ_PATTERN = r"PRJ-\d{6}"
ORG_PATTERN = r"ORG-\d{4}"
COUNT_ORG_SQL = "SELECT COUNT(*) FROM organizations"
# Install the packages listed in Setup into the selected notebook kernel.
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys
sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
import os
import tempfile
from pathlib import Path
notebook_temp = tempfile.TemporaryDirectory(prefix="sqlseed-notebook-01-")
work_dir = Path(notebook_temp.name)
os.environ["SQLSEED_CACHE_DIR"] = str(work_dir / "cache")
db_path = build(work_dir / "demo.db")

# Fail visibly if a generation only partially succeeds.
generation_checks = []
def check_result(result, expected_count):
    if result.errors or result.count != expected_count:
        raise RuntimeError(f"{result.table_name}: expected {expected_count}, wrote {result.count}; errors={result.errors}")
    generation_checks.append({"table": result.table_name, "count": result.count, "errors": list(result.errors)})
    print(f"Verified {result.table_name}: {result.count} rows; errors={result.errors}")
    return result

def check_results(results, config_path):
    config = sqlseed.load_config(str(config_path))
    expected = {table.name: table.count for table in config.tables}
    for result in results:
        check_result(result, expected[result.table_name])
    if {result.table_name for result in results} != set(expected):
        raise RuntimeError("Not every configured table produced a result")
    return results

def check_cli(result):
    if result.exit_code != 0:
        raise RuntimeError(result.output) from result.exception
    return result


print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

### 📍 Architecture Position

| Module | File | Core Class/Function |
|------|------|------------|
| Core Orchestration | `src/sqlseed/core/orchestrator/` | `DataOrchestrator.fill_table()` |

> Corresponding architecture diagram: [§2 Core Orchestration Flow (fill_table execution chain)](../../docs/architecture.zh-CN.md#2-核心编排流程fill_table-执行链路)

## 1. One Line of Code, Ten Thousand Rows — The Power of sqlseed

Core philosophy of sqlseed: **generate massive test data with a single line of code**. No need to write generation scripts or maintain SQL fixtures — sqlseed auto-infers table structure, intelligently selects generation strategies, and streams data into the database.

```python
result = sqlseed.fill("app.db", table="users", count=100_000)
```

In [ ]:
# Fill parent table (organizations) first, then child table (members)
# sqlseed needs parent table data to resolve foreign key references
check_result(fill(str(db_path), table="organizations", count=5), 5)

# One line of code, generate 100 member rows
result = check_result(fill(str(db_path), table="members", count=100), 100)
print(result)
# → GenerationResult(table=members, count=100, elapsed=..., speed=...)

With just this one line — sqlseed accomplished:

1. **Read schema** — auto-detect columns, types, constraints of `members` table
2. **Smart mapping** — `name` → real name, `email` → email address, `member_no` → unique ID
3. **Stream write** — batch insert 100 rows, auto-handle UNIQUE constraints
4. **Return result** — `GenerationResult` contains row count, elapsed time, speed, etc.

## 2. Inspect Database Structure

Before generating data, let's see what tables and columns the database has.

In [ ]:
import sqlite3

conn = sqlite3.connect(str(db_path))
tables = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()]
print(f"Tables ({len(tables)}): {tables}")

for table in tables:
    cols = conn.execute(f"PRAGMA table_info({table})").fetchall()
    col_names = [c[1] for c in cols]
    print(f"  {table}: {col_names}")

conn.close()

## 3. Zero-Config Fill in Detail

Behind the single `fill()` line above, sqlseed does a lot of work. Let's look in detail:

In [ ]:
result = check_result(fill(str(db_path), table="members", count=10), 10)
print(result)

### GenerationResult in Detail

The returned `GenerationResult` contains rich execution info:

In [ ]:
print(f"Table name: {result.table_name}")
print(f"Rows inserted: {result.count}")
print(f"Elapsed: {result.elapsed:.3f}s")
print(f"Speed: {result.rows_per_second:.2f} rows/s")
print(f"Batch count: {result.batch_count}")
print(f"Errors: {result.errors}")

### View Generated Data

In [ ]:
conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT member_id, member_no, name, email, org_code FROM members LIMIT 5").fetchall()

# Tabular output
print(f"{'ID':>4s}  {'member_no':<20s}  {'name':<20s}  {'email':<30s}  {'org_code':<10s}")
print('-' * 90)
for row in rows:
    print(f"{row[0]:>4d}  {row[1]:<20s}  {row[2]:<20s}  {row[3]:<30s}  {row[4]:<10s}")
conn.close()

## 4. Preview Data (Without Writing to DB)

`sqlseed.preview()` generates data but **does not write to the database**, suitable for debugging and verifying mapping results.

In [ ]:
# Fill projects table first (requires organizations to have data)
check_result(fill(str(db_path), table="projects", count=5), 5)

# View generated project info
conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT project_no, short_code, name, org_code FROM projects").fetchall()
print(f"{'project_no':<12s}  {'short_code':<8s}  {'name':<25s}  {'org_code':<10s}")
print('-' * 58)
for row in rows:
    print(f"{row[0]:<12s}  {str(row[1] or ''):<8s}  {row[2]:<25s}  {row[3]:<10s}")
conn.close()

# preview generates new data (no write), can use columns to override generation strategy
print("\npreview generates new data (columns override):")
rows = preview(str(db_path), table="projects", count=3,
               columns={"project_no": {"type": "pattern", "regex": PRJ_PATTERN},
                        "name": {"type": "company"}})
for row in rows:
    print(f"  {row.get('project_no'):<12s}  {row.get('name')}")

## 5. Context Manager

`sqlseed.connect()` returns a `DataOrchestrator` context manager, suitable for scenarios requiring multiple fills. It auto-cleans resources on exit.

In [ ]:
with connect(str(db_path), provider="mimesis", locale="en") as orch:
    r1 = check_result(orch.fill_table("tags", count=10), 10)
    r2 = check_result(orch.fill_table("tasks", count=50), 50)
    print(f"Tags: {r1.count} rows in {r1.elapsed:.3f}s")
    print(f"Tasks: {r2.count} rows in {r2.elapsed:.3f}s")

## 6. Core Parameters in Detail

### 6.1 count — Number of Rows

Controls the volume of data generated. For columns with UNIQUE constraints, sqlseed auto-backtracks to ensure no duplicates.

In [ ]:
import sqlite3

conn = sqlite3.connect(str(db_path))
existing_codes = [r[0] for r in conn.execute("SELECT org_code FROM organizations").fetchall()]

result = check_result(fill(str(db_path), table="organizations", count=3,
              columns={"org_code": {"type": "pattern", "regex": ORG_PATTERN},
                       "parent_code": {"type": "choice", "choices": existing_codes}}), 3)
print(f"Generated {result.count} organizations ({result.elapsed:.3f}s, {result.rows_per_second:.0f} rows/s)")

rows = conn.execute("SELECT org_code, name, parent_code FROM organizations").fetchall()
print(f"\n{'org_code':<12s}  {'name':<25s}  {'parent_code':<12s}")
print('-' * 52)
for row in rows:
    parent = row[2] or '(root)'
    print(f"{row[0]:<12s}  {row[1]:<25s}  {parent:<12s}")
conn.close()

### 6.2 provider — Data Provider

| Provider | Additional provider dependency | Values |
|---|---|---|
| `mimesis` | Optional Mimesis extra | Localized data methods |
| `faker` | None; Faker is required by Core | Localized data methods |
| `base` | None; built in | Placeholder strings and numbers |

Choose by locale and required format. The next cell prints real previews instead of assuming a fixed quality or speed ranking.

In [ ]:
for provider_name in ["base", "faker", "mimesis"]:
    rows = preview(str(db_path), table="members", count=2, provider=provider_name)
    sep = "=" * 60
    print()
    print(sep)
    print(f"  Provider: {provider_name}")
    print(sep)
    for row in rows:
        addr = str(row.get("address", "N/A"))[:40]
        print(f"  name={row.get('name', 'N/A'):<20s} email={row.get('email', 'N/A'):<30s}")
        print(f"  phone={row.get('phone', 'N/A'):<20s} address={addr}")

### 6.3 seed — Reproducibility

A fixed seed can reproduce previews with the same schema, existing data, provider and package versions. Appending or replaying against a changed database can change IDs and generated values.

In [ ]:
rows_a = preview(str(db_path), table="members", count=3, seed=42)
rows_b = preview(str(db_path), table="members", count=3, seed=42)
rows_c = preview(str(db_path), table="members", count=3, seed=99)

names_a = [r["name"] for r in rows_a]
names_b = [r["name"] for r in rows_b]
names_c = [r["name"] for r in rows_c]

print(f"seed=42 (run 1): {names_a}")
print(f"seed=42 (run 2): {names_b}")
print(f"seed=99:        {names_c}")
print(f"\nseed=42 reproducible: {names_a == names_b}")
print(f"Different seeds:  {names_a != names_c}")
assert names_a == names_b
assert names_a != names_c


### 6.4 batch_size — Batch Write Size

Controls the number of rows written per batch. Larger batch_size improves write performance but uses more memory.

- Default: 5000
- Small data volume (<1000 rows): batch_size has little impact
- Large data volume (>10K rows): increasing batch_size can improve speed

In [ ]:
# batch_size controls rows per batch, default 5000
# The configured batch size is an upper bound; progress display does not enlarge it.
r = check_result(fill(str(db_path), table="members", count=50), 50)
print(f"50 rows: {r.elapsed:.3f}s, {r.rows_per_second:.0f} rows/s")

r = check_result(fill(str(db_path), table="members", count=500), 500)
print(f"500 rows: {r.elapsed:.3f}s, {r.rows_per_second:.0f} rows/s")

### 6.5 enrich — Smart Enrichment Mode

When the database already has some data, `enrich=True` makes sqlseed analyze existing data patterns (e.g., enum values, value ranges) and keep them consistent when generating new data.

See architecture.md §2 enrich flow

In [ ]:
import sqlite3

# enrich demo: first check existing data in organizations table

conn = sqlite3.connect(str(db_path))
before = conn.execute(COUNT_ORG_SQL).fetchone()[0]
orgs = conn.execute("SELECT org_code, name FROM organizations").fetchall()
print(f"Existing data: {before} organizations")
for r in orgs:
    print(f"  {r[0]:<12s} {r[1]}")

# Get existing org_code as candidate values for parent_code
existing_codes = [r[0] for r in orgs]

# enrich=True: analyze existing data patterns, new data stays consistent
r2 = check_result(fill(str(db_path), table="organizations", count=5, enrich=True,
          columns={"org_code": {"type": "pattern", "regex": ORG_PATTERN},
                   "parent_code": {"type": "choice", "choices": existing_codes}}), 5)
after = conn.execute(COUNT_ORG_SQL).fetchone()[0]
new_rows = conn.execute("SELECT org_code, name, parent_code FROM organizations ORDER BY rowid DESC LIMIT 5").fetchall()
print(f"\nenrich added {r2.count} ({before} -> {after}):")
for r in new_rows:
    print(f"  {r[0]:<12s} {r[1]:<22s} parent={r[2] or ''}")
conn.close()


### 6.6 clear_before — Clear Then Fill

Clears this table before filling. For related tables, clear children before parents or start with a fresh database; filling parents first does not make a graph reset safe. This example clears `tags`, whose dependent tables are empty.

In [ ]:
result = check_result(fill(str(db_path), table="tags", count=8, clear_before=True), 8)
print(f"Cleared and refilled: {result.count} rows")

conn = sqlite3.connect(str(db_path))
count = conn.execute("SELECT COUNT(*) FROM tags").fetchone()[0]
print(f"Current row count: {count}")
conn.close()

## 7. Custom Column Mapping

Use the `columns` parameter to override auto-inferred column mapping:

In [ ]:
result = check_result(fill(
    str(db_path),
    table="members",
    count=5,
    columns={
        "name": {"type": "name"},               # real name
        "email": {"type": "email"},             # email address
        "phone": {"type": "phone"},             # phone number
        "balance": {"type": "float", "min_value": 100.0, "max_value": 500.0},  # specified range
        "is_active": {"type": "boolean"},       # boolean
    },
), 5)
print(f"Custom mapping: {result.count} rows")

conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT name, email, phone, balance, is_active FROM members ORDER BY member_id DESC LIMIT 5").fetchall()  # noqa: E501
print(f"\n{'name':<18s}  {'email':<28s}  {'phone':<18s}  {'balance':>8s}  {'active':>6s}")
print('-' * 85)
for row in rows:
    print(f"{row[0]:<18s}  {row[1]:<28s}  {row[2]:<18s}  {row[3]:>8.2f}  {row[4]!s:>6s}")
conn.close()

## 🎯 enrich Mode in Detail

When `enrich=True`, sqlseed auto-detects **enum columns** (e.g., `status`, `*_type`, `is_*`, etc.), and even if these columns have DEFAULT values or are nullable, it generates meaningful enum values instead of skipping them.

EnrichmentEngine uses 19 enum column name patterns and cardinality ratio calculation to identify enum columns. See [02-column-mapping](02-column-mapping.ipynb).

In [ ]:
import sqlite3

# enrich mode in detail: demo on organizations table
# First fill baseline data
check_result(fill(str(db_path), table="organizations", count=3, seed=42,
     columns={"org_code": {"type": "pattern", "regex": r"ENR-\d{4}"}}), 3)

conn = sqlite3.connect(str(db_path))
before = conn.execute(COUNT_ORG_SQL).fetchone()[0]
existing_codes = [r[0] for r in conn.execute("SELECT org_code FROM organizations").fetchall()]
print(f"Before enrich: {before} organizations")

# enrich adds organizations, auto-keeping existing data patterns
r2 = check_result(fill(str(db_path), table="organizations", count=3, enrich=True,
          columns={"org_code": {"type": "pattern", "regex": r"ENR-\d{4}"},
                   "parent_code": {"type": "choice", "choices": existing_codes}}), 3)
after = conn.execute(COUNT_ORG_SQL).fetchone()[0]
new_rows = conn.execute("SELECT org_code, name, parent_code FROM organizations ORDER BY rowid DESC LIMIT 3").fetchall()
print(f"After enrich: {after} organizations (added {r2.count})")
print("\nenrich added:")
for r in new_rows:
    print(f"  {r[0]:<12s} {r[1]:<22s} parent={r[2] or ''}")
conn.close()

## 📋 fill_from_config Getting Started

Simplest YAML config + one `fill_from_config()` call to batch-fill multiple tables. See [06-config-deep-dive](06-config-deep-dive.ipynb).

In [ ]:
from pathlib import Path
from sqlseed import fill_from_config
from sqlseed.config.loader import save_config
from sqlseed.config.models import GeneratorConfig, TableConfig

simple_config = GeneratorConfig(
    db_path=str(db_path),
    tables=[
        TableConfig(name="organizations", count=3),
    ]
)
config_path = (work_dir / "_quickstart_config.yaml")
save_config(simple_config, str(config_path))

results = check_results(fill_from_config(str(config_path)), str(config_path))
for r in results:
    print(f"  {r.table_name}: {r.count} rows in {r.elapsed:.3f}s")

config_path.unlink(missing_ok=True)

## ⚠️ Error Handling

Common errors and how to handle them:

In [ ]:
missing = fill(str(db_path), table="nonexistent_table", count=1)
assert missing.count == 0 and missing.errors
print("Expected missing-table error:", missing.errors)

try:
    fill(str(db_path), table="organizations", count=-1)
except ValueError as exc:
    print("Expected parameter error:", exc)
else:
    raise RuntimeError("Negative count must fail")

# Dangerous SQL identifier characters are rejected before database execution.
with sqlite3.connect(str(db_path)) as connection:
    before = connection.execute(COUNT_ORG_SQL).fetchone()[0]
try:
    fill(str(db_path), table="; DROP TABLE organizations; --", count=1)
except ValueError as exc:
    print("Expected identifier rejection:", exc)
else:
    raise RuntimeError("Dangerous identifier must be rejected")
with sqlite3.connect(str(db_path)) as connection:
    assert connection.execute(COUNT_ORG_SQL).fetchone()[0] == before



## 8. Summary

| API | Purpose | Writes to DB |
|-----|------|:----------:|
| `fill()` | One-line fill | ✅ |
| `preview()` | Preview without write | ❌ |
| `connect()` | Context manager | ✅ |
| `fill_from_config()` | YAML/JSON batch fill | ✅ |

| Parameter | Default | Description |
|------|--------|------|
| `count` | 1000 | Number of rows |
| `provider` | mimesis | Data provider (mimesis/faker/base) |
| `seed` | None | Random seed, reproducible when set |
| `batch_size` | 5000 | Batch write size |
| `enrich` | False | Smart enrichment mode |
| `clear_before` | False | Clear then fill |
| `locale` | en_US | Locale setting |

**Next**: [02-column-mapping.ipynb](02-column-mapping.ipynb) — Deep dive into the 9-level strategy chain

In [ ]:
# Verify exact database totals after the full notebook, plus every declared FK.
import sqlite3
expected_counts = {'organizations': 22, 'members': 665, 'projects': 5, 'tasks': 50, 'tags': 8, 'reviews': 0}
with sqlite3.connect(str(db_path)) as verification_db:
    actual_counts = {
        table: verification_db.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0]
        for table in expected_counts  # Fixed tutorial table names.
    }
    assert actual_counts == expected_counts, (actual_counts, expected_counts)
    fk_errors = verification_db.execute("PRAGMA foreign_key_check").fetchall()
    assert not fk_errors, fk_errors
print("Verified database row counts:", actual_counts)
print("Database FK check:", fk_errors)
print("Verified fill operations:", len(generation_checks))
